# Unit 13 / Chapter 13: Quantum Recommendation Systems

> **Main Learning Objective:** See how Netflix-style matrix-factorization recommenders work, what the original quantum recommendation algorithm (Kerenidis and Prakash) promised, and why Ewin Tang's dequantization changed the story. Build a mini quantum-inspired recommender.

| Section | Topic |
|---|---|
| 13.1 | Classical recommenders: matrix factorization in 3 minutes |
| 13.2 | The quantum recommendation algorithm (KP 2016) |
| 13.3 | The Tang dequantization and its lesson |
| 13.4 | Building a mini low-rank sampler |

---
## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100

# Tiny quantum simulator used across all units
def ket0(n):
    s = np.zeros(2**n, dtype=complex); s[0] = 1.0
    return s
def kron_all(mats):
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out
I2 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Y  = np.array([[0,-1j],[1j,0]], dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)
H  = (1/np.sqrt(2))*np.array([[1,1],[1,-1]], dtype=complex)
def Rx(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-1j*s],[-1j*s,c]], dtype=complex)
def Ry(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]], dtype=complex)
def Rz(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]], dtype=complex)
def apply_1q(gate, qubit, n):
    return kron_all([gate if i==qubit else I2 for i in range(n)])
def apply_cnot(control, target, n):
    dim = 2**n
    op = np.zeros((dim, dim), dtype=complex)
    for x in range(dim):
        bits = [(x >> (n-1-i)) & 1 for i in range(n)]
        if bits[control] == 1:
            bits[target] ^= 1
        y = 0
        for b in bits:
            y = (y<<1) | b
        op[y, x] = 1
    return op
def expZ(state, qubit, n):
    Zop = apply_1q(Z, qubit, n)
    return float(np.real(np.conj(state) @ Zop @ state))
print("Quantum simulator ready.")

---
## Course check-in

This logs that you started **Unit 13**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 13
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 13.1: Classical Recommenders in 3 Minutes

Netflix, Spotify, and Amazon all use variations on the same idea: a giant matrix R where rows are users, columns are items, and R[u, i] is the rating (or a proxy like "did the user watch it").

Most of R is empty, because most users have not rated most items. The goal is to fill in the missing entries so we can recommend items with high predicted ratings.

The dominant classical technique is **low-rank matrix factorization**:

  R ~ U * V^T,   where U is (users x k) and V is (items x k) with k tiny (say 10 to 100).

Each user gets a small feature vector, each item gets a small feature vector, and the predicted rating is the dot product. Training this via SGD is what took Netflix from grid search to recommendations that actually work.

In [ ]:
# Toy 5-user x 4-item rating matrix with rank ~2
np.random.seed(0)
n_users, n_items, k_true = 5, 4, 2
U_true = np.random.randn(n_users, k_true)
V_true = np.random.randn(n_items, k_true)
R = U_true @ V_true.T
R = np.round(R, 2)
print("Rating matrix R (users x items):")
print(R)

u_svd, s, vt = np.linalg.svd(R, full_matrices=False)
print("\nSingular values:", np.round(s, 2))
print("Rank of R:", np.sum(s > 1e-8))

---
# Section 13.2: The Quantum Recommendation Algorithm (KP 2016)

Kerenidis and Prakash (2016) proposed a quantum recommendation algorithm with a headline claim: recommending a good item for a user takes time **polylog(mn)**, where R is m x n, compared to poly(mn) for classical algorithms. That would be an exponential speedup.

The core ideas:

1. Assume access to R through a **QRAM** (Quantum RAM): a data structure that can prepare superpositions of R's rows in log time.
2. Use **quantum singular value estimation** to project a user's row onto the top-k singular components of R (its "taste directions").
3. Sample from that projected row to get a recommended item.

For years this was the poster child for quantum machine learning speedups.

---
# Section 13.3: The Tang Dequantization and Its Lesson

In 2018, an undergraduate at UT Austin named **Ewin Tang** stunned the field: she showed that if you allow the classical algorithm the *same* kind of sampling access (length-squared sampling from rows and columns of R), then a purely classical algorithm can recommend an item in polylog time too.

The quantum "speedup" was really an artifact of comparing quantum-with-QRAM to classical-without-sampling-access. Once the playing field was leveled, the exponential gap disappeared.

Since then, several other QML "speedups" have been dequantized the same way. The lesson: watch what data-access model each side gets. Real quantum speedups for practical ML tasks are harder to find than the 2016 excitement suggested.

Below we implement a small **length-squared sampler**, the classical trick that killed the KP speedup.

In [ ]:
def length_squared_sample(row, n_samples):
    probs = row**2
    probs = probs / probs.sum()
    return np.random.choice(len(row), size=n_samples, p=probs)

user0 = R[0]
print("user 0's row:", np.round(user0, 2))
samples = length_squared_sample(user0, 1000)
print("sampled item frequency (should track magnitude of each entry):")
for i in range(n_items):
    print(f"  item {i}: mag^2 = {(user0[i]**2 / (user0**2).sum()):.3f}, "
          f"empirical = {(samples == i).mean():.3f}")

### Activity 13.1

Sample from row 2 of R instead. Print which item gets picked most often and check that it matches the largest absolute value in that row.

In [ ]:
# TODO: sample from R[2] and identify the most likely item.
row2 = None            # <-- R[2]
samples2 = None        # <-- length_squared_sample(row2, 2000)

if samples2 is not None:
    counts = np.bincount(samples2, minlength=n_items)
    print("row 2:", np.round(row2, 2))
    print("sample counts:", counts.tolist())
    print("most sampled item:", int(np.argmax(counts)))
    print("largest-magnitude item:", int(np.argmax(np.abs(row2))))
else:
    print("Fill in row2 and samples2, then re-run.")

<details><summary>Solution</summary>

```python
row2 = R[2]
samples2 = length_squared_sample(row2, 2000)
```
The most sampled item should equal argmax(|row2|).
</details>

---
# Section 13.4: Building a Mini Low-Rank Sampler

A functional recommender needs one more step: project the user's row onto the top-k singular components of R before sampling, so that we recommend items in the direction of the user's actual "taste subspace" rather than just their biggest raw entry.

Below we implement this as a purely classical shadow of the KP idea. The point of running this is to see that low-rank projection plus length-squared sampling gives sensible recommendations.

In [ ]:
def rank_k_project(vec, R, k):
    _, s, vt = np.linalg.svd(R, full_matrices=False)
    V_k = vt[:k].T                       # items x k
    coords = vec @ V_k                   # k coords
    return V_k @ coords                  # projected item scores

user0_scores = rank_k_project(user0, R, k=2)
print("raw scores :", np.round(user0, 2))
print("proj scores:", np.round(user0_scores, 2))

samples = length_squared_sample(user0_scores, 5000)
counts = np.bincount(samples, minlength=n_items)
print("recommendation counts across 5000 samples:", counts.tolist())
print("top recommendation for user 0:", int(np.argmax(counts)))

### Activity 13.2

Recommend an item for user 4 using rank-2 projection + length-squared sampling.

In [ ]:
# TODO: recommend an item for user 4.
scores4 = None           # <-- rank_k_project(R[4], R, k=2)
samples4 = None          # <-- length_squared_sample(scores4, 5000)

if samples4 is not None:
    print("top recommendation for user 4:", int(np.argmax(np.bincount(samples4, minlength=n_items))))
else:
    print("Fill in scores4 and samples4, then re-run.")

<details><summary>Solution</summary>

```python
scores4 = rank_k_project(R[4], R, k=2)
samples4 = length_squared_sample(scores4, 5000)
```
</details>

---
## Section summary

* Classical recommenders are dominated by low-rank matrix factorization.
* The 2016 quantum recommendation algorithm claimed exponential speedup, assuming QRAM.
* Ewin Tang's 2018 dequantization showed that classical algorithms with the same sampling access are just as fast, killing the exponential speedup.
* Practical takeaway: any QML speedup claim needs a fair comparison to a classical algorithm with the equivalent data-access model.

---
## End-of-Unit Quiz (10 multiple choice)

**Q1.** Classical recommenders like Netflix's are usually built on:

A. Depth-first search
B. Low-rank matrix factorization
C. Regular expressions
D. Physical simulation

**Q2.** In R ~ U V^T, the small dimension k is called the:

A. Batch size
B. Latent factor count
C. Learning rate
D. Ratings scale

**Q3.** The KP quantum recommendation algorithm claimed a runtime of:

A. O(mn)
B. O(polylog(mn))
C. O(2^n)
D. O(n log n)

**Q4.** KP relied on which data structure to access the rating matrix?

A. QRAM
B. Hash map
C. Bloom filter
D. Skip list

**Q5.** Ewin Tang's contribution was to show that:

A. QRAM is impossible
B. A classical algorithm with sampling access matches the quantum runtime
C. Recommendations are unnecessary
D. Netflix should use quantum servers

**Q6.** Length-squared sampling from a vector v picks index i with probability:

A. v[i]
B. |v[i]|
C. v[i]^2 / sum(v^2)
D. 1 / len(v)

**Q7.** Projecting a user's row onto the top-k singular components of R gives:

A. The top-k rated items directly
B. A denoised score vector along the dominant taste directions
C. The average rating
D. A random unit vector

**Q8.** After Tang's dequantization, which of the following is true?

A. There is no more research in QML
B. Several other QML "speedups" have also been dequantized
C. Quantum computers are useless for recommendations
D. QRAM has been physically built

**Q9.** In our 5x4 toy problem, the true rank of R was:

A. 5
B. 4
C. 2
D. 1

**Q10.** The main practical lesson from the recommendation dequantization is:

A. QML has zero uses
B. Always compare quantum runtimes against classical algorithms with the equivalent data-access model
C. Only use QML for image tasks
D. Quantum recommenders will be deployed by 2026

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 13!")